In [ ]:
import torch
import torchio as tio
import numpy as np

images = np.arange(1, 36 * 3 + 1).astype(np.float32).reshape(2, 1, 6, 6)
masks = -np.arange(1, 36 * 3 + 1).astype(np.float32).reshape(2, 1, 6, 6)
images

array([[[[ 1.,  2.,  3.,  4.,  5.,  6.],
         [ 7.,  8.,  9., 10., 11., 12.],
         [13., 14., 15., 16., 17., 18.],
         [19., 20., 21., 22., 23., 24.],
         [25., 26., 27., 28., 29., 30.],
         [31., 32., 33., 34., 35., 36.]]],


       [[[37., 38., 39., 40., 41., 42.],
         [43., 44., 45., 46., 47., 48.],
         [49., 50., 51., 52., 53., 54.],
         [55., 56., 57., 58., 59., 60.],
         [61., 62., 63., 64., 65., 66.],
         [67., 68., 69., 70., 71., 72.]]]], dtype=float32)

In [57]:
masks

array([[[[ -1.,  -2.,  -3.,  -4.,  -5.,  -6.],
         [ -7.,  -8.,  -9., -10., -11., -12.],
         [-13., -14., -15., -16., -17., -18.],
         [-19., -20., -21., -22., -23., -24.],
         [-25., -26., -27., -28., -29., -30.],
         [-31., -32., -33., -34., -35., -36.]]],


       [[[-37., -38., -39., -40., -41., -42.],
         [-43., -44., -45., -46., -47., -48.],
         [-49., -50., -51., -52., -53., -54.],
         [-55., -56., -57., -58., -59., -60.],
         [-61., -62., -63., -64., -65., -66.],
         [-67., -68., -69., -70., -71., -72.]]]], dtype=float32)

In [65]:
from torch.utils.data import Dataset, DataLoader


class ToyDataset(Dataset):
    def __init__(self, images, masks):
        self.images = images
        self.masks = masks

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = torch.from_numpy(self.images[idx]).unsqueeze(-1).float()
        mask = torch.from_numpy(self.masks[idx]).unsqueeze(-1).float()

        subject = tio.Subject(
            image=tio.ScalarImage(tensor=img),
            mask=tio.LabelMap(tensor=mask),
        )
        return subject


dataset = ToyDataset(images, masks)
tio_dataset = tio.SubjectsDataset(dataset)

In [ ]:
import lightning as L


class MyDataModule(L.LightningDataModule):
    def __init__(self, images, masks, batch_size=2, patch_size=(3, 3, 1)):
        super().__init__()
        self.images = images
        self.masks = masks
        self.batch_size = batch_size
        self.patch_size = patch_size

    def setup(self, stage=None):
        self.train_dataset = ToyDataset(self.images, self.masks)
        self.val_dataset = ToyDataset(self.images, self.masks)
        self.tio_train = tio.SubjectsDataset(self.train_dataset)
        self.tio_val = tio.SubjectsDataset(self.val_dataset)

    def train_dataloader(self):
        train_queue = tio.Queue(
            self.tio_train,
            max_length=100,  # Number of patches stored in memory
            samples_per_volume=self.batch_size,
            sampler=tio.data.UniformSampler(self.patch_size),
            num_workers=4,
        )

        return torch.utils.data.DataLoader(
            train_queue,
            batch_size=self.batch_size,
            shuffle=True,
        )

    def val_dataloader(self):
        return torch.utils.data.DataLoader(
            self.tio_val,
            batch_size=None,
        )


dm = MyDataModule(images, masks)
dm.setup()

In [77]:
for batch in dm.train_dataloader():
    print(batch["image"][tio.DATA].squeeze(-1))
    print("=" * 50)
    print(batch["mask"][tio.DATA].squeeze(-1))
    break

tensor([[[[43., 44., 45.],
          [49., 50., 51.],
          [55., 56., 57.]]],


        [[[22., 23., 24.],
          [28., 29., 30.],
          [34., 35., 36.]]]])
tensor([[[[-43., -44., -45.],
          [-49., -50., -51.],
          [-55., -56., -57.]]],


        [[[-22., -23., -24.],
          [-28., -29., -30.],
          [-34., -35., -36.]]]])


In [94]:
for batch in dm.val_dataloader():
    print(batch["image"][tio.DATA])
    print(batch["image"][tio.DATA].shape)
    print("=" * 50)
    print(batch["mask"][tio.DATA].squeeze(-1))
    print(batch["image"][tio.DATA].shape)
    break


tensor([[[[ 1.],
          [ 2.],
          [ 3.],
          [ 4.],
          [ 5.],
          [ 6.]],

         [[ 7.],
          [ 8.],
          [ 9.],
          [10.],
          [11.],
          [12.]],

         [[13.],
          [14.],
          [15.],
          [16.],
          [17.],
          [18.]],

         [[19.],
          [20.],
          [21.],
          [22.],
          [23.],
          [24.]],

         [[25.],
          [26.],
          [27.],
          [28.],
          [29.],
          [30.]],

         [[31.],
          [32.],
          [33.],
          [34.],
          [35.],
          [36.]]]])
torch.Size([1, 6, 6, 1])
tensor([[[ -1.,  -2.,  -3.,  -4.,  -5.,  -6.],
         [ -7.,  -8.,  -9., -10., -11., -12.],
         [-13., -14., -15., -16., -17., -18.],
         [-19., -20., -21., -22., -23., -24.],
         [-25., -26., -27., -28., -29., -30.],
         [-31., -32., -33., -34., -35., -36.]]])
torch.Size([1, 6, 6, 1])


In [106]:
sampler = tio.data.GridSampler(
    batch,
    patch_size=(3, 3, 1),
    patch_overlap=(2, 2, 0),
)

patch_loader = torch.utils.data.DataLoader(sampler, batch_size=16)
aggregator = tio.data.GridAggregator(sampler)
for patch_batch in patch_loader:
    imgs = patch_batch["image"][tio.DATA]
    print(imgs.squeeze(-1))
    print("=" * 50)

    aggregator.add_batch(imgs, patch_batch[tio.LOCATION])

## Prediction Reconstruction
output = aggregator.get_output_tensor()  # 🔥 KEY LINE
output.squeeze(-1)

tensor([[[[ 1.,  2.,  3.],
          [ 7.,  8.,  9.],
          [13., 14., 15.]]],


        [[[ 2.,  3.,  4.],
          [ 8.,  9., 10.],
          [14., 15., 16.]]],


        [[[ 3.,  4.,  5.],
          [ 9., 10., 11.],
          [15., 16., 17.]]],


        [[[ 4.,  5.,  6.],
          [10., 11., 12.],
          [16., 17., 18.]]],


        [[[ 7.,  8.,  9.],
          [13., 14., 15.],
          [19., 20., 21.]]],


        [[[ 8.,  9., 10.],
          [14., 15., 16.],
          [20., 21., 22.]]],


        [[[ 9., 10., 11.],
          [15., 16., 17.],
          [21., 22., 23.]]],


        [[[10., 11., 12.],
          [16., 17., 18.],
          [22., 23., 24.]]],


        [[[13., 14., 15.],
          [19., 20., 21.],
          [25., 26., 27.]]],


        [[[14., 15., 16.],
          [20., 21., 22.],
          [26., 27., 28.]]],


        [[[15., 16., 17.],
          [21., 22., 23.],
          [27., 28., 29.]]],


        [[[16., 17., 18.],
          [22., 23., 24.],
          [

tensor([[[ 1.,  2.,  3.,  4.,  5.,  6.],
         [ 7.,  8.,  9., 10., 11., 12.],
         [13., 14., 15., 16., 17., 18.],
         [19., 20., 21., 22., 23., 24.],
         [25., 26., 27., 28., 29., 30.],
         [31., 32., 33., 34., 35., 36.]]])

> Ojo con LabelSampler para el caso de desbalance de clases. Necesario para samplear parches donde efectivamente hay cosas que es necesario detectar. Necesario para el DeadTrees.